### Weather + UK data

In [ ]:
%pip install skl2onnx onnx onnxruntime


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

In [17]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.multioutput import MultiOutputRegressor


In [18]:
DEMAND_PATH = "/Users/sm_aswin21/Desktop/hack-europe/hackeurope26/ml/data/uk/uk-data/demand.csv"
GEN_PATH = "/Users/sm_aswin21/Desktop/hack-europe/hackeurope26/ml/data/uk/uk-data/generation.csv"
WEATHER_PATH = "/Users/sm_aswin21/Desktop/hack-europe/hackeurope26/ml/data/weather-data/uk_london_2016-02-15_2026-02-15_hourly.csv"



In [19]:
uk_demand = pd.read_csv(DEMAND_PATH)
uk_generation = pd.read_csv(GEN_PATH)
uk_weather = pd.read_csv(WEATHER_PATH)

In [20]:
print(uk_weather.columns)
print(uk_generation.head())
print(uk_demand.head())

Index(['country', 'country_code', 'location', 'time', 'temperature_2m',
       'relative_humidity_2m', 'precipitation', 'wind_speed_10m',
       'wind_direction_10m', 'surface_pressure', 'cloud_cover',
       'latitude_used', 'longitude_used', 'timezone'],
      dtype='object')
  Dataset           PublishTime             StartTime SettlementDate  \
0  FUELHH  2016-01-07T23:30:00Z  2016-01-07T23:00:00Z     2016-01-07   
1  FUELHH  2016-01-07T23:30:00Z  2016-01-07T23:00:00Z     2016-01-07   
2  FUELHH  2016-01-07T23:30:00Z  2016-01-07T23:00:00Z     2016-01-07   
3  FUELHH  2016-01-07T23:30:00Z  2016-01-07T23:00:00Z     2016-01-07   
4  FUELHH  2016-01-07T23:30:00Z  2016-01-07T23:00:00Z     2016-01-07   

   SettlementPeriod FuelType  Generation  
0                47     CCGT        8648  
1                47     COAL        3973  
2                47    INTEW         282  
3                47    INTFR        1496  
4                47   INTIRL         186  
  RecordType             Start

In [ ]:
uk_demand

In [21]:
# Merge Demand + Generation
gen_agg = (
    uk_generation
    .groupby("StartTime")["Generation"]
    .sum()
    .reset_index()
)

df = uk_demand.merge(gen_agg, on="StartTime", how="inner")
df = df.rename(columns={"Generation": "Total_Generation"})
df["Gap"] = df["Total_Generation"] - df["Demand"]

df["StartTime"] = pd.to_datetime(df["StartTime"])
df = df.sort_values("StartTime")



# Prepare weather + align datetime

uk_weather = uk_weather.rename(columns={"time": "StartTime"})
uk_weather["StartTime"] = pd.to_datetime(uk_weather["StartTime"])

# Resample demand/gen/gap to hourly (mean)
cols = ["Demand", "Total_Generation", "Gap"]

df_hourly = (
    df.set_index("StartTime")[cols]
      .resample("H")
      .mean()
      .reset_index()
)

# Normalize timezone handling (same as your code)
df_hourly["StartTime"] = pd.to_datetime(df_hourly["StartTime"], utc=True).dt.tz_convert(None)
uk_weather["StartTime"] = pd.to_datetime(uk_weather["StartTime"], utc=True).dt.tz_convert(None)

# Merge hourly df with weather
df_final = df_hourly.merge(uk_weather, on="StartTime", how="left")

# Remove missing values
df_final = df_final.dropna().reset_index(drop=True)
print(df_final.shape)


(86934, 17)


In [32]:
# Feature Engineering

def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["hour"] = df["StartTime"].dt.hour
    df["day_of_week"] = df["StartTime"].dt.dayofweek
    df["month"] = df["StartTime"].dt.month
    df["day_of_year"] = df["StartTime"].dt.dayofyear
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    # Cyclical encoding
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    return df


def add_lag_rolling_features(
    df: pd.DataFrame,
    target_col: str = "Gap",
    extra_series_cols=("Demand", "Total_Generation"),
    lags=(1, 2, 3, 6, 12, 24, 48, 168),
    rolls=(3, 6, 24, 168)
) -> pd.DataFrame:
    """
    Creates leak-free lag/rolling features.
    Rolling features are based on shift(1) so they only use the past.
    """
    df = df.copy()

    # Lags for target
    for L in lags:
        df[f"{target_col}_lag_{L}"] = df[target_col].shift(L)

    # Rolling stats for target (shift by 1 to avoid leakage)
    shifted = df[target_col].shift(1)
    for W in rolls:
        df[f"{target_col}_roll_mean_{W}"] = shifted.rolling(W).mean()
        df[f"{target_col}_roll_std_{W}"] = shifted.rolling(W).std()
        df[f"{target_col}_roll_min_{W}"] = shifted.rolling(W).min()
        df[f"{target_col}_roll_max_{W}"] = shifted.rolling(W).max()

    # Lags + rolling for Demand and Generation
    for col in extra_series_cols:
        for L in lags:
            df[f"{col}_lag_{L}"] = df[col].shift(L)

        shifted_col = df[col].shift(1)
        for W in rolls:
            df[f"{col}_roll_mean_{W}"] = shifted_col.rolling(W).mean()
            df[f"{col}_roll_std_{W}"] = shifted_col.rolling(W).std()

    # Differences (ramps)
    df[f"{target_col}_diff_1"] = df[target_col].diff(1)
    df[f"{target_col}_diff_24"] = df[target_col].diff(24)

    # Weather change features
    if "temperature_2m" in df.columns:
        df["temp_diff_1"] = df["temperature_2m"].diff(1)
        df["temp_diff_24"] = df["temperature_2m"].diff(24)

    return df


def make_supervised(df: pd.DataFrame, horizon: int = 1, target_col: str = "Gap") -> pd.DataFrame:
    """
    Predict Gap at t + horizon using features at time t.
    """
    df = df.copy()
    df["y"] = df[target_col].shift(-horizon)
    df = df.dropna(subset=["y"])
    return df

# Single-horizon model (1-step ahead by default)

def rmse(y_true, y_pred) -> float:
    return np.sqrt(mean_squared_error(y_true, y_pred))


def mape(y_true, y_pred, eps: float = 1e-6) -> float:
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100

WEATHER_COLS = [
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "wind_speed_10m", "wind_direction_10m", "surface_pressure", "cloud_cover",
    "latitude_used", "longitude_used"   # confirmed present in your weather CSV
]

TIME_COLS_CYCLIC = [
    "hour_sin", "hour_cos", "dow_sin", "dow_cos",
    "month_sin", "month_cos", "day_of_year", "is_weekend"
]


In [33]:
# PART 1 — SINGLE HORIZON MODEL (1-step ahead)

df_feat = add_time_features(df_final)
df_feat = add_lag_rolling_features(df_feat)

HORIZON = 1
df_feat = make_supervised(df_feat, horizon=HORIZON, target_col="Gap")
df_feat = df_feat.dropna().reset_index(drop=True)

lag_roll_cols_single = [
    c for c in df_feat.columns
    if ("_lag_" in c) or ("_roll_" in c) or ("_diff_" in c)
]

feature_cols_single = WEATHER_COLS + TIME_COLS_CYCLIC + lag_roll_cols_single
feature_cols_single = [c for c in feature_cols_single if c in df_feat.columns]

X_single = df_feat[feature_cols_single]
y_single = df_feat["y"]

print(f"\n[Single-Horizon] X: {X_single.shape}, y: {y_single.shape}")



[Single-Horizon] X: (86765, 77), y: (86765,)


In [34]:
missing_info = X_single.isnull().sum()
missing_pct  = (missing_info / len(X_single)) * 100
missing_df   = pd.DataFrame({"Count": missing_info, "Pct": missing_pct}).query("Count > 0")
if not missing_df.empty:
    print("Missing values:\n", missing_df)



In [35]:
# TimeSeries CV
TEST_SIZE = 30 * 24
tscv = TimeSeriesSplit(n_splits=5, test_size=TEST_SIZE)

model_single = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("reg", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=500,
        random_state=42
    ))
])

fold_results = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_single), start=1):
    X_train, X_test = X_single.iloc[train_idx], X_single.iloc[test_idx]
    y_train, y_test = y_single.iloc[train_idx], y_single.iloc[test_idx]

    model_single.fit(X_train, y_train)
    preds = model_single.predict(X_test)

    fold_results.append({
        "fold":   fold,
        "MAE":    mean_absolute_error(y_test, preds),
        "RMSE":   rmse(y_test, preds),
        "R2":     r2_score(y_test, preds),
        "MAPE_%": mape(y_test.values, preds)
    })

results_single = pd.DataFrame(fold_results)
print("\n[Single-Horizon] CV Results:")
print(results_single)
print("\nMean:", results_single[["MAE", "RMSE", "R2", "MAPE_%"]].mean())
print("Std: ", results_single[["MAE", "RMSE", "R2", "MAPE_%"]].std())



[Single-Horizon] CV Results:
   fold         MAE        RMSE        R2      MAPE_%
0     1  338.343708  465.627362  0.792742   33.220203
1     2  362.799197  484.210166  0.860285  105.408979
2     3  431.189660  578.476135  0.840256   40.837168
3     4  359.875776  505.851135  0.772382   60.029217
4     5  392.249841  530.623787  0.896717   48.588957

Mean: MAE       376.891636
RMSE      512.957717
R2          0.832477
MAPE_%     57.616905
dtype: float64
Std:  MAE       35.910406
RMSE      43.937003
R2         0.050376
MAPE_%    28.496334
dtype: float64


In [36]:
print("\nRefitting single-horizon model on full data...")
model_single.fit(X_single, y_single)

initial_type_single = [("float_input", FloatTensorType([None, X_single.shape[1]]))]
onnx_single = convert_sklearn(model_single, initial_types=initial_type_single, target_opset=17)

with open("gap_single_horizon.onnx", "wb") as f:
    f.write(onnx_single.SerializeToString())
print("Saved gap_single_horizon.onnx")

# Sanity check
sess_single = rt.InferenceSession("gap_single_horizon.onnx")
sample_single = X_single.iloc[:3].values.astype(np.float32)
preds_single  = sess_single.run(None, {sess_single.get_inputs()[0].name: sample_single})
print(f"Single-horizon ONNX output shape: {np.array(preds_single).shape}")  # expect (1, 3)




Refitting single-horizon model on full data...
Saved gap_single_horizon.onnx
Single-horizon ONNX output shape: (1, 3, 1)


# PART 2 — MULTI HORIZON MODEL (1–12 steps ahead)


In [37]:
HORIZONS = range(1, 13)

df_model = df_final.sort_values("StartTime").reset_index(drop=True).copy()

# Add time features BEFORE snapshotting independent_cols
df_model = add_time_features(df_model)
independent_cols = df_model.columns.tolist()

# Create future targets
for h in HORIZONS:
    df_model[f"Gap_t_plus_{h}h"] = df_model["Gap"].shift(-h)

dependent_cols = [c for c in df_model.columns if c not in independent_cols]
print(f"\n[Multi-Horizon] Target columns ({len(dependent_cols)}): {dependent_cols}")

# Add lag/roll features (using same helper, no leakage)
def add_lags(df: pd.DataFrame, col: str, lags=(1, 2, 3, 6, 12, 24, 48, 168)) -> pd.DataFrame:
    df = df.copy()
    for L in lags:
        df[f"{col}_lag_{L}"] = df[col].shift(L)
    return df




[Multi-Horizon] Target columns (12): ['Gap_t_plus_1h', 'Gap_t_plus_2h', 'Gap_t_plus_3h', 'Gap_t_plus_4h', 'Gap_t_plus_5h', 'Gap_t_plus_6h', 'Gap_t_plus_7h', 'Gap_t_plus_8h', 'Gap_t_plus_9h', 'Gap_t_plus_10h', 'Gap_t_plus_11h', 'Gap_t_plus_12h']


In [38]:

def add_rolls(df: pd.DataFrame, col: str, windows=(3, 6, 24, 168)) -> pd.DataFrame:
    df = df.copy()
    s = df[col].shift(1)
    for W in windows:
        df[f"{col}_roll_mean_{W}"] = s.rolling(W).mean()
        df[f"{col}_roll_std_{W}"]  = s.rolling(W).std()
    return df

df_model = add_lags(df_model, "Gap")
df_model = add_rolls(df_model, "Gap")
df_model = add_lags(df_model, "Demand")
df_model = add_lags(df_model, "Total_Generation")

# Differences
df_model["Gap_diff_1"]  = df_model["Gap"].diff(1)
df_model["Gap_diff_24"] = df_model["Gap"].diff(24)
if "temperature_2m" in df_model.columns:
    df_model["temp_diff_1"]  = df_model["temperature_2m"].diff(1)
    df_model["temp_diff_24"] = df_model["temperature_2m"].diff(24)

# Use cyclic time features (consistent with single-horizon model)
TIME_COLS_MULTI = TIME_COLS_CYCLIC  # hour_sin/cos, dow_sin/cos, etc.

lag_roll_cols_multi = [
    c for c in df_model.columns
    if ("_lag_" in c) or ("_roll_" in c) or ("_diff_" in c)
]

feature_cols_multi = WEATHER_COLS + TIME_COLS_MULTI + lag_roll_cols_multi
feature_cols_multi = [c for c in feature_cols_multi if c in df_model.columns]

# Drop NaNs
data = pd.concat([df_model[feature_cols_multi], df_model[dependent_cols], df_model[["StartTime"]]], axis=1)
data = data.dropna().reset_index(drop=True)

X_multi = data[feature_cols_multi]
Y_multi = data[dependent_cols]
times   = data["StartTime"]

print(f"[Multi-Horizon] X: {X_multi.shape}, Y: {Y_multi.shape}")

# TimeSeries CV
tscv = TimeSeriesSplit(n_splits=5, test_size=TEST_SIZE)

base_estimator = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=8,
    max_iter=500,
    random_state=42
)
model_multi = MultiOutputRegressor(base_estimator)

rows = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X_multi), start=1):
    X_train, X_test = X_multi.iloc[train_idx], X_multi.iloc[test_idx]
    Y_train, Y_test = Y_multi.iloc[train_idx], Y_multi.iloc[test_idx]

    model_multi.fit(X_train, Y_train)
    Y_pred = model_multi.predict(X_test)

    for j, h in enumerate(HORIZONS):
        y_true_h = Y_test.iloc[:, j].values
        y_pred_h = Y_pred[:, j]

        rows.append({
            "fold":      fold,
            "horizon_h": h,
            "MAE":       mean_absolute_error(y_true_h, y_pred_h),
            "RMSE":      rmse(y_true_h, y_pred_h),
            "R2":        r2_score(y_true_h, y_pred_h),
        })

results_multi = pd.DataFrame(rows)

print("\n[Multi-Horizon] Per-horizon mean across folds:")
print(results_multi.groupby("horizon_h")[["MAE", "RMSE", "R2"]].mean())
print("\nOverall mean:")
print(results_multi[["MAE", "RMSE", "R2"]].mean())

try:
    display(
        results_multi.groupby("horizon_h")[["MAE", "RMSE", "R2"]].mean()
        .style
        .background_gradient(subset=["MAE", "RMSE"], cmap="Reds_r", axis=0)
        .background_gradient(subset=["R2"],           cmap="Greens",  axis=0)
        .format({"MAE": "{:.2f}", "RMSE": "{:.2f}", "R2": "{:.3f}"})
    )
except NameError:
    pass

# Refit on all data + export
print("\nRefitting multi-horizon model on full data...")
model_multi.fit(X_multi, Y_multi)

initial_type_multi = [("float_input", FloatTensorType([None, X_multi.shape[1]]))]
onnx_multi = convert_sklearn(model_multi, initial_types=initial_type_multi, target_opset=17)

with open("gap_multi_horizon.onnx", "wb") as f:
    f.write(onnx_multi.SerializeToString())
print("Saved gap_multi_horizon.onnx")

# Sanity check
sess_multi  = rt.InferenceSession("gap_multi_horizon.onnx")
sample_multi = X_multi.iloc[:3].values.astype(np.float32)
preds_multi  = sess_multi.run(None, {sess_multi.get_inputs()[0].name: sample_multi})
print(f"Multi-horizon ONNX output shape: {np.array(preds_multi).shape}")  # expect (1, 3, 12) or (3, 12)


[Multi-Horizon] X: (86754, 53), Y: (86754, 12)

[Multi-Horizon] Per-horizon mean across folds:
                  MAE         RMSE        R2
horizon_h                                   
1          377.710204   513.763728  0.833197
2          564.723770   749.782286  0.655179
3          683.037104   898.825588  0.510263
4          763.990302   991.577293  0.410505
5          826.914997  1067.375975  0.320989
6          869.504996  1119.024965  0.254402
7          898.311135  1150.158574  0.209216
8          925.060505  1191.659173  0.151709
9          945.594281  1216.356467  0.119141
10         948.215725  1226.308425  0.106090
11         961.313789  1241.887542  0.088510
12         963.759715  1247.363320  0.079630

Overall mean:
MAE      810.678044
RMSE    1051.173611
R2         0.311569
dtype: float64


,MAE,RMSE,R2
horizon_h,,,
1,377.71,513.76,0.833
2,564.72,749.78,0.655
3,683.04,898.83,0.510
4,763.99,991.58,0.411
5,826.91,1067.38,0.321
6,869.50,1119.02,0.254
7,898.31,1150.16,0.209
8,925.06,1191.66,0.152
9,945.59,1216.36,0.119



Refitting multi-horizon model on full data...
Saved gap_multi_horizon.onnx
Multi-horizon ONNX output shape: (1, 3, 12)


In [23]:
# Feature sets
weather_cols = [
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "wind_speed_10m", "wind_direction_10m", "surface_pressure", "cloud_cover"
]

time_cols = [
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "day_of_year", "is_weekend"
]


In [24]:
lag_roll_cols = [c for c in df_feat.columns if ("_lag_" in c) or ("_roll_" in c) or ("_diff_" in c)]

feature_cols = weather_cols + time_cols + lag_roll_cols
feature_cols = [c for c in feature_cols if c in df_feat.columns]

X = df_feat[feature_cols]
y = df_feat["y"]

print(X.shape, y.shape)

(86933, 77) (86933,)


In [25]:
# TimeSeries CV + model
TEST_SIZE = 30 * 24  # 30 days in hourly data
tscv = TimeSeriesSplit(n_splits=5, test_size=TEST_SIZE)

model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("reg", HistGradientBoostingRegressor(
        learning_rate=0.05,
        max_depth=8,
        max_iter=500,
        random_state=42
    ))
])

fold_results = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    fold_results.append({
        "fold": fold,
        "MAE": mean_absolute_error(y_test, preds),
        "RMSE": rmse(y_test, preds),
        "R2": r2_score(y_test, preds),
        "MAPE_%": mape(y_test.values, preds)
    })

results_df = pd.DataFrame(fold_results)
print(results_df)
print("\nMean across folds:")
print(results_df[["MAE", "RMSE", "R2", "MAPE_%"]].mean())
print("\nStd across folds:")
print(results_df[["MAE", "RMSE", "R2", "MAPE_%"]].std())



   fold         MAE        RMSE        R2      MAPE_%
0     1  341.309931  468.435398  0.790235   34.342023
1     2  360.437129  483.410440  0.860746  107.752853
2     3  431.840085  576.237626  0.841490   39.715893
3     4  356.119171  496.564057  0.780663   57.902444
4     5  398.226388  539.110057  0.893387   52.188954

Mean across folds:
MAE       377.586541
RMSE      512.751515
R2          0.833304
MAPE_%     58.380433
dtype: float64

Std across folds:
MAE       36.879510
RMSE      44.193070
R2         0.047582
MAPE_%    29.165086
dtype: float64


In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType


In [27]:
HORIZONS = range(1, 13)

df_model = df_final.sort_values("StartTime").reset_index(drop=True).copy()

independent_cols = df_model.columns.tolist()

# Create future targets
for h in HORIZONS:
    df_model[f"Gap_t_plus_{h}h"] = df_model["Gap"].shift(-h)

# Dependent columns
dependent_cols = [c for c in df_model.columns if c not in independent_cols]
print("Dependent columns:", dependent_cols[:5], "...", dependent_cols[-1])

def add_lags(df: pd.DataFrame, col: str = "Gap", lags=(1, 2, 3, 6, 12, 24, 48, 168)) -> pd.DataFrame:
    df = df.copy()
    for L in lags:
        df[f"{col}_lag_{L}"] = df[col].shift(L)
    return df

def add_rolls(df: pd.DataFrame, col: str = "Gap", windows=(3, 6, 24, 168)) -> pd.DataFrame:
    df = df.copy()
    s = df[col].shift(1)  # shift(1) avoids leakage
    for W in windows:
        df[f"{col}_roll_mean_{W}"] = s.rolling(W).mean()
        df[f"{col}_roll_std_{W}"] = s.rolling(W).std()
    return df


# Add lag/roll features
df_model = add_lags(df_model, "Gap")
df_model = add_rolls(df_model, "Gap")

# lag Demand/Generation
df_model = add_lags(df_model, "Demand")
df_model = add_lags(df_model, "Total_Generation")

# Feature columns 
weather_cols = [
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "wind_speed_10m", "wind_direction_10m", "surface_pressure", "cloud_cover"
]
time_cols = ["hour", "day_of_week", "month"]
geo_cols = ["latitude_used", "longitude_used"]  

lag_roll_cols = [c for c in df_model.columns if ("_lag_" in c) or ("_roll_" in c)]
feature_cols = weather_cols + time_cols + geo_cols + lag_roll_cols
feature_cols = [c for c in feature_cols if c in df_model.columns]

X = df_model[feature_cols]
Y = df_model[dependent_cols]

# Drop NaNs due to lags/rolling + future shifts
data = pd.concat([X, Y, df_model[["StartTime"]]], axis=1).dropna().reset_index(drop=True)

X = data[feature_cols]
Y = data[dependent_cols]
times = data["StartTime"]  


Dependent columns: ['Gap_t_plus_1h', 'Gap_t_plus_2h', 'Gap_t_plus_3h', 'Gap_t_plus_4h', 'Gap_t_plus_5h'] ... Gap_t_plus_12h


In [29]:

# MultiOutput model + TimeSeries CV
TEST_SIZE = 30 * 24
tscv = TimeSeriesSplit(n_splits=5, test_size=TEST_SIZE)

base = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=8,
    max_iter=500,
    random_state=42
)
model = MultiOutputRegressor(base)

rows = []
for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    Y_train, Y_test = Y.iloc[train_idx], Y.iloc[test_idx]

    model.fit(X_train, Y_train)
    
    Y_pred = model.predict(X_test)

    for j, h in enumerate(HORIZONS):
        y_true_h = Y_test.iloc[:, j].values
        y_pred_h = Y_pred[:, j]

        rows.append({
            "fold": fold,
            "horizon_h": h,
            "MAE": mean_absolute_error(y_true_h, y_pred_h),
            "RMSE": rmse(y_true_h, y_pred_h),
            "R2": r2_score(y_true_h, y_pred_h),
        })

results = pd.DataFrame(rows)

print("Per-horizon mean across folds:")
print(results.groupby("horizon_h")[["MAE", "RMSE", "R2"]].mean())

print("\nOverall mean (all horizons & folds):")
print(results[["MAE", "RMSE", "R2"]].mean())

try:
    display(
        results.groupby("horizon_h")[["MAE", "RMSE", "R2"]].mean()
        .style
        .background_gradient(subset=["MAE", "RMSE"], cmap="Reds_r", axis=0)
        .background_gradient(subset=["R2"], cmap="Greens", axis=0)
        .format({"MAE": "{:.2f}", "RMSE": "{:.2f}", "R2": "{:.3f}"})
    )
except NameError:
    # If not running in a notebook with display()
    pass

Per-horizon mean across folds:
                  MAE         RMSE        R2
horizon_h                                   
1          573.655565   763.028550  0.637880
2          683.889043   892.882262  0.508833
3          761.258301   992.554300  0.401154
4          835.749516  1081.904238  0.296230
5          884.245712  1137.333418  0.225477
6          923.476458  1186.166854  0.159891
7          935.757495  1207.273123  0.129076
8          947.578067  1225.000732  0.107830
9          964.415112  1246.206944  0.080519
10         963.927131  1255.007066  0.072200
11         969.516054  1254.794664  0.074262
12         972.001530  1264.730168  0.060077

Overall mean (all horizons & folds):
MAE      867.955832
RMSE    1125.573527
R2         0.229452
dtype: float64


,MAE,RMSE,R2
horizon_h,,,
1,573.66,763.03,0.638
2,683.89,892.88,0.509
3,761.26,992.55,0.401
4,835.75,1081.90,0.296
5,884.25,1137.33,0.225
6,923.48,1186.17,0.160
7,935.76,1207.27,0.129
8,947.58,1225.00,0.108
9,964.42,1246.21,0.081


In [ ]:
model.fit(X, Y)

initial_type = [("float_input", FloatTensorType([None, X.shape[1]]))]
onnx_model = convert_sklearn(model, initial_types=initial_type, target_opset=17)

with open("gap_multi_horizon.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())
print("Saved gap_multi_horizon.onnx")
